# HOG size vs. log2FC. 

Load packages

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

Load the annotated result dataset from Salmon map

In [ ]:
salmon_map_full_annot = pd.read_csv("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis/salmon_map_dominance_DE_sex_results_new_filtering.csv", float_precision='legacy')

# Replace the zeros in padj with 1e-308 avoid log10 issues
salmon_map_full_annot["padj_safe"] = salmon_map_full_annot["padj"].replace(0, 1e-308).fillna(1)

# Add the negative log 10 padj for plotting
salmon_map_full_annot["neglog10_padj"] = -np.log10(salmon_map_full_annot["padj_safe"])

# Add significance to differentially expressed genes 
salmon_map_full_annot["significant"] = (
    (salmon_map_full_annot["padj"] < 0.05) &
    (salmon_map_full_annot["log2FoldChange"].abs() > 1)
)

# Add a label to the significant genes
salmon_map_full_annot["label"] = salmon_map_full_annot["gene_id"].where(salmon_map_full_annot["significant"], "")

salmon_map_full_annot

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,start,...,GOs,EC,KEGG_ko,KEGG_Pathway,COG_category,eggNOG_OGs,padj_safe,neglog10_padj,significant,label
0,248.420188,0.692248,0.131010,5.283945,1.264310e-07,2.436236e-07,g2.t1,g2,utg000001l,220384.0,...,-,-,ko:K02273,"ko00190,ko01100,ko04260,ko04714,ko04932,ko0501...",I,"2FBM0@1|root,2TCUK@2759|Eukaryota,398E5@33154|...",2.436236e-07,6.613281,False,
1,92.726112,0.655816,0.169767,3.863035,1.119872e-04,1.816552e-04,g3.t1,g3,utg000001l,227675.0,...,"GO:0003674,GO:0005488,GO:0005515,GO:0005543,GO...",3.1.26.5,"ko:K14529,ko:K17543","ko03013,map03013",DUZ,"KOG2101@1|root,KOG2101@2759|Eukaryota,396Y9@33...",1.816552e-04,3.740752,False,
2,136.044931,-0.620457,0.178648,-3.473064,5.145527e-04,7.936587e-04,g4.t1,g4,utg000001l,245866.0,...,"GO:0001101,GO:0001666,GO:0003674,GO:0003824,GO...",2.5.1.61,ko:K01749,"ko00860,ko01100,ko01110,ko01120,map00860,map01...",H,"COG0181@1|root,KOG2892@2759|Eukaryota,38D6W@33...",7.936587e-04,3.100366,False,
3,381.084979,-0.916813,0.069608,-13.171077,1.287480e-39,7.390355e-39,g6.t1,g6,utg000001l,263441.0,...,-,-,-,-,B,"2E6V3@1|root,2SDHR@2759|Eukaryota",7.390355e-39,38.131335,False,
4,70.718212,-1.042894,0.092940,-11.221090,3.212849e-29,1.373292e-28,g7.t1,g7,utg000001l,371755.0,...,"GO:0002682,GO:0002683,GO:0002685,GO:0002686,GO...",-,-,-,Z,"COG4886@1|root,KOG0532@2759|Eukaryota,38HCP@33...",1.373292e-28,27.862237,True,g7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14159,10.439363,0.155829,1.106835,0.140788,8.880376e-01,8.997925e-01,g34647.t1,g34647,utg003542l,22333.0,...,-,-,-,-,T,"COG4886@1|root,KOG4641@2759|Eukaryota,39T87@33...",8.997925e-01,0.045858,False,
14160,10.044300,0.631293,0.230505,2.738741,6.167502e-03,8.691213e-03,g34695.t1,g34695,utg003603l,1.0,...,"GO:0000003,GO:0000578,GO:0000932,GO:0001709,GO...",-,ko:K17597,-,KU,"KOG3732@1|root,KOG3732@2759|Eukaryota,38HNW@33...",8.691213e-03,2.060920,False,
14161,20.275098,0.380299,0.210397,1.807533,7.067915e-02,8.783567e-02,g34843.t1,g34843,utg003648l,5656.0,...,NaN,NaN,NaN,NaN,NaN,NaN,8.783567e-02,1.056329,False,
14162,8.286716,2.238606,0.365799,6.119773,9.370857e-10,2.009399e-09,g34922.t1,g34922,utg003714l,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2.009399e-09,8.696934,True,g34922


Remove entries that does not have a HOG

In [ ]:
salmon_map_results_HOG = salmon_map_full_annot.dropna(subset=["HOG"])


Compute number of paralogs in each HOG

In [ ]:
hog_size = (
    salmon_map_results_HOG
    .groupby("HOG")
    .size()
    .reset_index(name="HOG_size")
)

Add the hog_size column to the results table by merging on HOG name

In [ ]:
salmon_map_results_HOG = salmon_map_results_HOG.merge(
    hog_size,
    on="HOG",
    how="left"
)

salmon_map_results_HOG
# Down from 14.164 to 12.837 transcripts

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,start,...,EC,KEGG_ko,KEGG_Pathway,COG_category,eggNOG_OGs,padj_safe,neglog10_padj,significant,label,HOG_size
0,248.420188,0.692248,0.131010,5.283945,1.264310e-07,2.436236e-07,g2.t1,g2,utg000001l,220384.0,...,-,ko:K02273,"ko00190,ko01100,ko04260,ko04714,ko04932,ko0501...",I,"2FBM0@1|root,2TCUK@2759|Eukaryota,398E5@33154|...",2.436236e-07,6.613281,False,,2
1,92.726112,0.655816,0.169767,3.863035,1.119872e-04,1.816552e-04,g3.t1,g3,utg000001l,227675.0,...,3.1.26.5,"ko:K14529,ko:K17543","ko03013,map03013",DUZ,"KOG2101@1|root,KOG2101@2759|Eukaryota,396Y9@33...",1.816552e-04,3.740752,False,,2
2,136.044931,-0.620457,0.178648,-3.473064,5.145527e-04,7.936587e-04,g4.t1,g4,utg000001l,245866.0,...,2.5.1.61,ko:K01749,"ko00860,ko01100,ko01110,ko01120,map00860,map01...",H,"COG0181@1|root,KOG2892@2759|Eukaryota,38D6W@33...",7.936587e-04,3.100366,False,,2
3,381.084979,-0.916813,0.069608,-13.171077,1.287480e-39,7.390355e-39,g6.t1,g6,utg000001l,263441.0,...,-,-,-,B,"2E6V3@1|root,2SDHR@2759|Eukaryota",7.390355e-39,38.131335,False,,1
4,70.718212,-1.042894,0.092940,-11.221090,3.212849e-29,1.373292e-28,g7.t1,g7,utg000001l,371755.0,...,-,-,-,Z,"COG4886@1|root,KOG0532@2759|Eukaryota,38HCP@33...",1.373292e-28,27.862237,True,g7,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12832,57.411980,-0.987138,0.285952,-3.452106,5.562288e-04,8.559830e-04,g34609.t1,g34609,utg003498l,40065.0,...,-,ko:K04638,"ko05016,map05016",T,"KOG0972@1|root,KOG0972@2759|Eukaryota,38NJ2@33...",8.559830e-04,3.067535,False,,3
12833,309.335990,-0.776463,0.193966,-4.003094,6.251939e-05,1.031381e-04,g34610.t1,g34610,utg003498l,51899.0,...,-,ko:K02890,"ko03010,map03010",J,"COG0091@1|root,KOG1711@2759|Eukaryota,39N6Y@33...",1.031381e-04,3.986581,False,,3
12834,10.439363,0.155829,1.106835,0.140788,8.880376e-01,8.997925e-01,g34647.t1,g34647,utg003542l,22333.0,...,-,-,-,T,"COG4886@1|root,KOG4641@2759|Eukaryota,39T87@33...",8.997925e-01,0.045858,False,,1
12835,20.275098,0.380299,0.210397,1.807533,7.067915e-02,8.783567e-02,g34843.t1,g34843,utg003648l,5656.0,...,NaN,NaN,NaN,NaN,NaN,8.783567e-02,1.056329,False,,3


Plot each transcript vs. HOG size 

In [ ]:
#Add some jitter on x-axis
np.random.seed(42)  # for reproducibility

jitter = np.random.uniform(
    -0.15, 0.15,
    size=len(salmon_map_results_HOG)
)

fig = px.scatter(
    x=salmon_map_results_HOG["HOG_size"] + jitter,
    y=salmon_map_results_HOG["log2FoldChange"],
    opacity=0.4,
    labels={
        "x": "HOG size (number of paralogs)",
        "y": "Transcript log2FoldChange"
    },
    title="Transcript-level sex bias as a function of HOG size"
)

fig.add_hline(y=0, line_dash="dash")
fig.show()


Try with absolute values instead of directional.  
Add abs

In [ ]:
salmon_map_results_HOG["abs_log2FC"] = (
    salmon_map_results_HOG["log2FoldChange"].abs()
)

Plot abs bias vs. HOG size 

In [ ]:
fig = px.scatter(
    salmon_map_results_HOG,
    x=salmon_map_results_HOG["HOG_size"] + jitter,
    y=salmon_map_results_HOG["log2FoldChange"],
    hover_name="transcript_id",
    opacity=0.4,
    labels={
        "x": "HOG size (number of paralogs)",
        "y": "Transcript log2FoldChange"
    },
    title="Transcript-level absolute sex bias vs. HOG size"
)

fig.show()


Variance within each HOG size group. Group by HOG_size instead of HOG

In [ ]:
variance_by_size = (
    salmon_map_results_HOG
    .groupby("HOG_size")["log2FoldChange"]
    .agg(
        variance="var",
        sd="std",
        n="count"
    )
    .reset_index()
)
variance_by_size


,HOG_size,variance,sd,n
0,1,2.524564,1.588888,7568
1,2,2.553843,1.598075,3032
2,3,3.075588,1.753736,888
3,4,2.540290,1.593829,400
4,5,1.957932,1.399261,230
5,6,3.429901,1.851999,180
6,7,2.268202,1.506055,77
7,8,4.233507,2.057549,168
8,9,2.711837,1.646766,90
9,10,0.076325,0.276270,10


Plot variance vs HOG Size 

In [ ]:
fig = px.line(
    variance_by_size,
    x="HOG_size",
    y="sd",
    markers=True,
    labels={
        "HOG_size": "HOG size (number of paralogs)",
        "sd": "SD of log2FoldChange"
    },
    title="Variance in sex-biased expression amaon paralogs by HOG size"
)

fig.show()
Do within hogs, plot against hog size. 

# Blocks below for averages within HOGs:

Compute mean sex bias in each HOG

In [ ]:
#Do absolute instead of directional? 
hog_lfc = (
    salmon_map_results_HOG
    .groupby("HOG")["log2FoldChange"]
    .mean()
    .reset_index(name="mean_log2FC")
)

Merge

In [ ]:
hog_summary = hog_lfc.merge(hog_size, on="HOG")
hog_summary

,HOG,mean_log2FC,HOG_size
0,N0.HOG0000007,1.660752,8
1,N0.HOG0000008,0.939406,8
2,N0.HOG0000009,1.453381,4
3,N0.HOG0000010,0.590586,12
4,N0.HOG0000011,-1.429319,1
...,...,...,...
9608,N0.HOG0024541,-1.486879,2
9609,N0.HOG0024543,0.481861,2
9610,N0.HOG0024547,-0.151882,1
9611,N0.HOG0024673,2.180027,1


Plot

In [ ]:
#Add some jitter on x-axis
np.random.seed(42)  # for reproducibility

hog_summary["HOG_size_jitter"] = (
    hog_summary["HOG_size"]
    + np.random.uniform(-0.15, 0.15, size=len(hog_summary))
)


fig = px.scatter(
    hog_summary,
    x="HOG_size_jitter",
    y="mean_log2FC",
    hover_name="HOG",
    opacity=0.6,
    labels={
        "HOG_size_jitter": "HOG size (number of paralogs)",
        "mean_log2FC": "Mean log2FoldChange"
    },
    title="Paralog number vs sex-biased expression (Directional)"
)

fig.add_hline(y=0, line_dash="dash")
fig.show()


Try with absolute L2FC

In [ ]:
hog_lfc_abs = (
    salmon_map_results_HOG
    .assign(abs_log2FC=lambda x: x["log2FoldChange"].abs())
    .groupby("HOG")["abs_log2FC"]
    .mean()
    .reset_index(name="mean_abs_log2FC")
)

hog_abs_summary = hog_lfc_abs.merge(hog_size, on="HOG")


In [ ]:
#Add some jitter on x-axis
np.random.seed(42)  # for reproducibility

hog_abs_summary["HOG_size_jitter"] = (
    hog_abs_summary["HOG_size"]
    + np.random.uniform(-0.15, 0.15, size=len(hog_abs_summary))
)

fig = px.scatter(
    hog_abs_summary,
    x="HOG_size_jitter",
    y="mean_abs_log2FC",
    hover_name="HOG",
    opacity=0.6,
    labels={
        "HOG_size_jitter": "HOG size (number of paralogs)",
        "mean_abs_log2FC": "Mean |log2FoldChange|"
    },
    title="Paralog number vs strength of sex-biased expression (Absolute)"
)

fig.show()


# Male-bias only and female-bias only

In [ ]:
#split each sex up based on log2fc
male_biased = salmon_map_results_HOG[
    salmon_map_results_HOG["log2FoldChange"] > 0
]

female_biased = salmon_map_results_HOG[
    salmon_map_results_HOG["log2FoldChange"] < 0
]


In [ ]:
# male-biased
hog_male = (
    male_biased
    .groupby("HOG")["log2FoldChange"]
    .mean()
    .reset_index(name="mean_male_log2FC")
)


In [ ]:
#female biased
hog_female = (
    female_biased
    .assign(abs_log2FC=lambda x: x["log2FoldChange"].abs())
    .groupby("HOG")["abs_log2FC"]
    .mean()
    .reset_index(name="mean_female_log2FC")
)


In [ ]:
#hog size same as before
hog_size = (
    salmon_map_results_HOG
    .groupby("HOG")
    .size()
    .reset_index(name="HOG_size")
)


In [ ]:
#merge all
hog_sex_bias = (
    hog_size
    .merge(hog_male, on="HOG", how="left")
    .merge(hog_female, on="HOG", how="left")
)


hog_sex_bias


,HOG,HOG_size,mean_male_log2FC,mean_female_log2FC
0,N0.HOG0000007,8,1.660752,NaN
1,N0.HOG0000008,8,1.131950,0.408400
2,N0.HOG0000009,4,1.453381,NaN
3,N0.HOG0000010,12,0.689358,0.495909
4,N0.HOG0000011,1,NaN,1.429319
...,...,...,...,...
9608,N0.HOG0024541,2,NaN,1.486879
9609,N0.HOG0024543,2,0.481861,NaN
9610,N0.HOG0024547,1,NaN,0.151882
9611,N0.HOG0024673,1,2.180027,NaN


plot male biased transcripts

In [ ]:
#Add some jitter on x-axis
np.random.seed(42)  # for reproducibility

hog_sex_bias["HOG_size_jitter"] = (
    hog_sex_bias["HOG_size"]
    + np.random.uniform(-0.15, 0.15, size=len(hog_sex_bias))
)

fig = px.scatter(
    hog_sex_bias,
    x="HOG_size_jitter",
    y="mean_male_log2FC",
    opacity=0.6,
    labels={
        "HOG_size_jitter": "HOG size",
        "mean_male_log2FC": "Mean male-biased log2FC"
    },
    title="Paralog number vs male-biased expression strength"
)
fig.show()


Plot female baised transcripts

In [ ]:
fig = px.scatter(
    hog_sex_bias,
    x="HOG_size_jitter",
    y="mean_female_log2FC",
    opacity=0.6,
    labels={
        "HOG_size_jitter": "HOG size",
        "mean_female_log2FC": "Mean female-biased |log2FC|"
    },
    title="Paralog number vs female-biased expression strength"
)
fig.show()
